# Aula de Reforço — DevOps até a Semana 7

**Unidade curricular:** Integração e Entrega Contínua — DevOps  
**Foco:** fundamentos, Git/GitFlow, CI/CD, testes automatizados e Quality Gates  
**Prática principal:** executar um ciclo GitFlow completo em um repositório local

> Esta aula foi planejada para revisão orientada e prática individual. Os comandos usam apenas um repositório local, portanto podem ser repetidos sem conta no GitHub.


## Objetivos de aprendizagem

Ao final, você deverá conseguir:

- explicar DevOps por meio de cultura, fluxo, feedback e melhoria contínua;
- relacionar CALMS, métricas DORA e redução do tempo de entrega;
- distinguir Git, GitHub, CI, Continuous Delivery e Continuous Deployment;
- escolher a branch correta no GitFlow para feature, release e hotfix;
- escrever commits atômicos com Conventional Commits e versões SemVer;
- relacionar testes unitários, de integração e E2E com Quality Gates;
- executar e explicar um fluxo GitFlow completo, incluindo conflito e tag.


## Roteiro da aula

1. Diagnóstico inicial
2. Visão integrada das semanas 1 a 7
3. Git e governança do histórico
4. CI/CD, testes e Quality Gates
5. Laboratório GitFlow local
6. Desafio e verificação de aprendizagem


## 1. Diagnóstico inicial

Responda sem consultar o material:

1. Qual problema organizacional o DevOps tenta reduzir?
2. Uma feature deve nascer de qual branch no GitFlow?
3. Qual diferença existe entre Continuous Delivery e Continuous Deployment?
4. Um teste que chama uma rota HTTP e usa um banco temporário pertence a qual camada?
5. O que deve acontecer quando um Quality Gate falha?

Guarde as respostas e compare com o gabarito ao final.


## 2. Mapa integrado das semanas 1 a 7

| Semana | Conceito central | Evidência de aprendizagem |
|---|---|---|
| 1 | DevOps, CALMS, fluxo de valor e DORA | Identificar gargalos e ciclos de feedback |
| 2 | Git, DAG, branches, commits e GitFlow | Produzir histórico rastreável |
| 3 | Branches e ambientes de deploy | Relacionar mudança, preview e produção |
| 4 | GitFlow colaborativo e conflitos | Integrar trabalho sem perder alterações |
| 5 | Pirâmide de testes e Quality Gates | Selecionar o teste adequado e bloquear falhas |
| 6 | Aplicação no Projeto Integrador | Levar práticas ao projeto da equipe |
| 7 | Integração, E2E e GitHub Actions | Automatizar validações em PRs e pushes |


## 3. Fundamentos de DevOps

DevOps combina cultura e práticas de engenharia para reduzir o intervalo entre uma ideia e uma mudança confiável em produção. O foco recai sobre o sistema de trabalho inteiro, e não apenas sobre uma ferramenta.

### CALMS

- **Culture:** responsabilidade compartilhada, colaboração e post-mortem sem culpados.
- **Automation:** build, testes, provisionamento e deploy repetíveis.
- **Lean:** lotes pequenos, pouco trabalho em andamento e remoção de desperdícios.
- **Measurement:** decisões apoiadas por dados do fluxo e da operação.
- **Sharing:** conhecimento e aprendizado circulam entre pessoas e equipes.

### Três maneiras

- **Fluxo:** acelerar a passagem da mudança até o usuário.
- **Feedback:** detectar problemas cedo e devolver informação rapidamente.
- **Aprendizado:** experimentar com segurança e melhorar o processo continuamente.


### Métricas DORA

| Métrica | Pergunta que responde | Tendência desejada |
|---|---|---|
| Deployment Frequency | Com que frequência entregamos? | maior |
| Lead Time for Changes | Quanto tempo uma mudança leva até produção? | menor |
| Change Failure Rate | Quantas mudanças causam falha? | menor |
| Time to Restore Service | Quanto tempo levamos para recuperar o serviço? | menor |

As métricas devem orientar melhoria do sistema. Usá-las para punir indivíduos tende a incentivar manipulação e reduzir colaboração.


In [ ]:
def diagnostico_dora(deploys_mes, lead_time_h, falhas, deploys, mttr_h):
    taxa_falha = (falhas / deploys * 100) if deploys else 0
    return {
        "deploys_por_mes": deploys_mes,
        "lead_time_horas": lead_time_h,
        "change_failure_rate_pct": round(taxa_falha, 1),
        "mttr_horas": mttr_h,
    }

diagnostico_dora(deploys_mes=8, lead_time_h=30, falhas=2, deploys=20, mttr_h=4)


**Reflexão:** qual prática atacaria primeiro o maior gargalo do cenário acima? Justifique usando uma métrica e um pilar CALMS.


## 4. Git como grafo de histórico

Git registra snapshots ligados por relações de ancestralidade. Uma branch é um nome móvel que aponta para um commit. Um merge cria uma integração entre histórias, enquanto o rebase reaplica commits sobre uma nova base e reescreve seus identificadores.

### Regras úteis

- prefira commits pequenos, coerentes e testáveis;
- revise com `git status` e `git diff --staged` antes do commit;
- não faça rebase de commits públicos que colegas já utilizam;
- preserve branches protegidas e integre por Pull Request em equipes;
- use `git log --graph --oneline --all` para ler o histórico.


### Conventional Commits e SemVer

| Mudança | Commit típico | Efeito esperado na versão |
|---|---|---|
| Correção compatível | `fix(api): trata tarefa inexistente` | PATCH: 1.4.0 → 1.4.1 |
| Nova funcionalidade compatível | `feat(api): adiciona filtro por status` | MINOR: 1.4.1 → 1.5.0 |
| Mudança incompatível | `feat(api)!: altera contrato de resposta` | MAJOR: 1.5.0 → 2.0.0 |

Outros tipos frequentes: `docs`, `test`, `refactor`, `build`, `ci` e `chore`. A mensagem melhora a comunicação, mas o conteúdo do commit ainda precisa ser atômico.


## 5. GitFlow: branches e responsabilidades

| Branch | Origem | Destino ao concluir | Uso |
|---|---|---|---|
| `main` | permanente | — | estado de produção e tags |
| `develop` | permanente | — | integração da próxima versão |
| `feature/*` | `develop` | `develop` | funcionalidade isolada |
| `release/*` | `develop` | `main` e `develop` | estabilização de uma versão |
| `hotfix/*` | `main` | `main` e `develop` | correção urgente em produção |

GitFlow favorece ciclos formais de release e múltiplas linhas de manutenção. Branches longas aumentam divergência e custo de merge. Em entregas muito frequentes, Trunk-Based Development com branches curtas e feature flags pode ser mais adequado.


## 6. CI/CD e arquitetura de pipeline

- **Integração Contínua (CI):** cada integração dispara build e verificações automáticas para detectar regressões cedo.
- **Entrega Contínua (Continuous Delivery):** a mudança aprovada fica pronta para produção, mas uma decisão humana controla o deploy final.
- **Implantação Contínua (Continuous Deployment):** toda mudança que passa pelos gates segue automaticamente para produção.

Em GitHub Actions, um evento aciona um **workflow**. O workflow contém **jobs**, cada job executa em um **runner**, e cada job possui **steps** sequenciais. Secrets guardam credenciais fora do código.


### Exemplo conceitual de Quality Gate

`pull_request` → instalar dependências → lint → testes unitários → integração → E2E → cobertura mínima → liberar merge

Um gate falhou? O pipeline deve retornar erro e a proteção da branch deve impedir o merge. O valor está no bloqueio verificável, não apenas no relatório.


## 7. Pirâmide de testes

| Camada | Escopo | Velocidade | Quantidade esperada | Exemplo |
|---|---|---|---|---|
| Unitário | função ou classe isolada | alta | muitos | regra de conclusão de tarefa |
| Integração | componentes reais integrados | média | moderada | rota Flask com banco temporário |
| E2E | fluxo do usuário pelo sistema | baixa | poucos | criar, consultar e concluir tarefa |

Mocks, stubs e fixtures ajudam no isolamento e na preparação de cenários. Cobertura mede código executado pelos testes, mas não prova que os testes verificam o comportamento correto. Um limiar de cobertura deve funcionar junto com testes relevantes, lint e revisão.


## 8. Laboratório GitFlow local

### Pré-requisitos e segurança

- Git instalado: `git --version`
- execute os comandos em uma pasta neutra, fora de outro repositório;
- o laboratório cria a pasta `lab-gitflow-reforco`; se ela já existir, escolha outro nome;
- configure nome e e-mail apenas no repositório do laboratório.

> No Jupyter, comandos iniciados por `!` rodam no shell. Para aprender o fluxo, recomenda-se usar um terminal e marcar cada etapa concluída.


### Etapa 1 — Inicialização e branches permanentes

```bash
mkdir lab-gitflow-reforco
cd lab-gitflow-reforco
git init -b main
git config user.name "Aluno DevOps"
git config user.email "aluno@example.local"
printf "# Catálogo de tarefas\n" > README.md
git add README.md
git commit -m "docs: inicia catálogo de tarefas"
git switch -c develop
git log --graph --oneline --all
```

**Checkpoint:** `main` e `develop` apontam para o mesmo commit inicial.


### Etapa 2 — Feature e merge explícito

```bash
git switch -c feature/filtro-status develop
printf "\n- Filtro por status: planejado\n" >> README.md
git add README.md
git commit -m "feat(catalogo): adiciona filtro por status"
git switch develop
git merge --no-ff feature/filtro-status -m "merge: integra filtro por status"
git branch -d feature/filtro-status
git log --graph --oneline --decorate --all
```

**Checkpoint:** o merge `--no-ff` registra explicitamente o limite da feature no grafo. Em equipes, a integração ocorreria por Pull Request e revisão.


### Etapa 3 — Conflito controlado

Criaremos duas alterações concorrentes na mesma linha.

```bash
git switch -c feature/titulo-curto develop
printf "# Tarefas\n\n- Filtro por status: planejado\n" > README.md
git add README.md
git commit -m "docs: simplifica título do catálogo"

git switch develop
printf "# Catálogo DevOps\n\n- Filtro por status: planejado\n" > README.md
git add README.md
git commit -m "docs: contextualiza título do catálogo"

git merge feature/titulo-curto
```

O último comando deve sinalizar conflito. Isso é esperado.


### Etapa 4 — Resolução do conflito

1. Execute `git status`.
2. Abra `README.md` e localize `<<<<<<<`, `=======` e `>>>>>>>`.
3. Escolha um título final, por exemplo `# Catálogo de tarefas DevOps`.
4. Remova todos os marcadores e salve.
5. Conclua:

```bash
git add README.md
git commit -m "merge: resolve conflito no título do catálogo"
git branch -d feature/titulo-curto
git log --graph --oneline --decorate --all
```

Se quiser cancelar a tentativa antes de resolver, use `git merge --abort`.


### Etapa 5 — Release e tag SemVer

```bash
git switch -c release/1.0.0 develop
printf "\nVersão: 1.0.0\n" >> README.md
git add README.md
git commit -m "chore(release): prepara versão 1.0.0"

git switch main
git merge --no-ff release/1.0.0 -m "release: publica versão 1.0.0"
git tag -a v1.0.0 -m "Versão 1.0.0"

git switch develop
git merge --no-ff release/1.0.0 -m "merge: retorna release 1.0.0 para develop"
git branch -d release/1.0.0
git log --graph --oneline --decorate --all
```

**Checkpoint:** a tag `v1.0.0` aponta para a versão publicada em `main`, e a preparação da versão também está em `develop`.


### Etapa 6 — Hotfix e propagação

```bash
git switch main
git switch -c hotfix/1.0.1
printf "\nCorreção: normalização de status.\n" >> README.md
git add README.md
git commit -m "fix(catalogo): normaliza status das tarefas"

git switch main
git merge --no-ff hotfix/1.0.1 -m "hotfix: publica correção 1.0.1"
git tag -a v1.0.1 -m "Versão 1.0.1"

git switch develop
git merge --no-ff hotfix/1.0.1 -m "merge: propaga hotfix 1.0.1 para develop"
git branch -d hotfix/1.0.1
git log --graph --oneline --decorate --all
git tag --list
```

**Checkpoint:** o hotfix nasceu de `main`, gerou PATCH e voltou para `develop`.


### Etapa 7 — Auditoria do fluxo

Execute e explique cada saída:

```bash
git status
git branch --list
git tag --list
git log --graph --oneline --decorate --all
git show v1.0.1 --stat
```

Critérios de sucesso:

- árvore sem mudanças pendentes;
- branches permanentes `main` e `develop`;
- tags `v1.0.0` e `v1.0.1`;
- commits com mensagens significativas;
- hotfix presente nas duas branches permanentes.


## 9. Do laboratório local ao trabalho em equipe

No repositório remoto, adapte o fluxo:

1. publique a branch temporária com `git push -u origin nome-da-branch`;
2. abra um Pull Request para o destino correto;
3. aguarde revisão e execução do CI;
4. corrija falhas na mesma branch;
5. conclua o merge somente com gates aprovados;
6. sincronize a branch local e remova branches temporárias.

Evite enviar diretamente para `main` ou `develop` quando a equipe utiliza proteção de branches. Nunca publique credenciais.


## 10. Desafio prático

A partir de `develop`, implemente uma nova feature chamada `feature/prioridade`. O arquivo deve ganhar uma linha que descreva a prioridade das tarefas.

Entregáveis:

- um commit `feat` atômico;
- merge `--no-ff` em `develop`;
- captura ou cópia do grafo final;
- uma frase explicando por que a mudança gera MINOR e não PATCH;
- proposta de três verificações que um pipeline deveria executar antes do merge.


## 11. Verificação de aprendizagem

1. Qual pilar CALMS sustenta pipelines repetíveis?
2. Por que branches muito longas prejudicam a integração contínua?
3. De onde nasce uma `release/*` e para onde ela retorna?
4. Por que o hotfix precisa voltar para `develop`?
5. Qual diferença operacional separa Delivery de Deployment?
6. Por que muitos testes E2E tornam a suíte lenta e frágil?
7. Cobertura de 80% prova qualidade? Explique.
8. Qual mecanismo impede o merge quando o CI falha?


## 12. Gabarito comentado

1. **Automation.** Automatização reduz variação manual e torna o processo repetível.
2. Elas acumulam divergência, atrasam feedback e aumentam conflitos.
3. Nasce de `develop`; ao concluir, integra em `main` e `develop`.
4. Para evitar que a correção desapareça da próxima versão.
5. Delivery mantém uma aprovação humana para produção; Deployment publica automaticamente após os gates.
6. E2E atravessa muitos componentes, exige mais infraestrutura e oferece diagnóstico menos localizado.
7. Não. Cobertura mede execução, não a qualidade das asserções ou a relevância dos cenários.
8. Uma regra de proteção da branch que exige checks obrigatórios aprovados.

**Diagnóstico inicial:** respostas esperadas: reduzir silos e conflitos de objetivos; `feature/*` nasce de `develop`; Delivery tem decisão humana antes de produção; rota HTTP com banco temporário é teste de integração; gate falho bloqueia a progressão.


## 13. Síntese

O fluxo coerente liga cultura e engenharia:

**mudança pequena → branch adequada → commit rastreável → Pull Request → testes automatizados → Quality Gate → release versionada → feedback operacional**

GitFlow organiza responsabilidades, mas não substitui comunicação, revisão e automação. O objetivo final permanece o mesmo: entregar mudanças pequenas com segurança e aprender rapidamente com o resultado.


## Materiais relacionados no repositório

- `Aulas/semana_01.ipynb` a `Aulas/semana_07.ipynb`
- `Materiais/guia_passo_a_passo_gitflow.md`
- `Materiais/padrao_de_commits.md`
- `Materiais/versionamento_semantico.md`
- `Materiais/todo-app/` para testes unitários, integração e E2E
